<a href="https://colab.research.google.com/github/daffaidf/Introduction-to-ai/blob/main/2024071024_MuhammadDaffaRinaldi_Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Case study 1**
 **Problem Definition**

Business Problem:
Memprediksi apakah siswa akan lulus (G3 ≥ 10) atau tidak.

Machine Learning Task:
Binary Classification

Target:
1 = Lulus
0 = Tidak Lulus

In [4]:
from google.colab import files
uploaded = files.upload()

Saving student-mat.csv to student-mat.csv


In [8]:
import pandas as pd

df = pd.read_csv("student-mat.csv")  # tanpa sep

print(df.shape)
print(df.columns)
df.head()

(395, 33)
Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [9]:
df["pass"] = (df["G3"] >= 10).astype(int)

df["pass"].value_counts()

,count
pass,
1,265
0,130


In [10]:
X = df.drop(columns=["G3", "pass"])
y = df["pass"]

In [11]:
print(X.isnull().sum().sort_values(ascending=False))

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
dtype: int64


In [12]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [16]:
def check_distribution(name, y):
    print(f"\nDistribusi {name}")
    print(y.value_counts())
    print(y.value_counts(normalize=True))

check_distribution("TRAIN", y_train)
check_distribution("VALIDATION", y_val)
check_distribution("TEST", y_test)


Distribusi TRAIN
pass
1    185
0     91
Name: count, dtype: int64
pass
1    0.67029
0    0.32971
Name: proportion, dtype: float64

Distribusi VALIDATION
pass
1    40
0    19
Name: count, dtype: int64
pass
1    0.677966
0    0.322034
Name: proportion, dtype: float64

Distribusi TEST
pass
1    40
0    20
Name: count, dtype: int64
pass
1    0.666667
0    0.333333
Name: proportion, dtype: float64


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# FIT hanya TRAIN (ini penting banget)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

print("Shape setelah preprocessing:")
print("Train:", X_train_prep.shape)
print("Val:", X_val_prep.shape)
print("Test:", X_test_prep.shape)

Shape setelah preprocessing:
Train: (276, 58)
Val: (59, 58)
Test: (60, 58)


# case study 2
Business Problem:
Mendeteksi transaksi kartu kredit yang termasuk fraud secara otomatis.

Machine Learning Task:
Binary Classification (Highly Imbalanced)

Target:
1 = Fraud
0 = Non-Fraud

Catatan:
Dataset sangat tidak seimbang sehingga accuracy bukan metrik utama.

In [20]:
print("Shape:", df.shape)
df.head()
df.info()

Shape: (395, 34)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    objec

In [21]:
import pandas as pd

DATA_PATH = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [22]:
target_col = "Class"

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

print("Rasio Fraud:", y.mean())

Rasio Fraud: 0.001727485630620034


In [23]:
print(X.isnull().sum().sort_values(ascending=False))

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
dtype: int64


In [24]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Jumlah kolom numerik:", len(num_cols))

Jumlah kolom numerik: 30


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [26]:
def check_distribution(name, y):
    print(f"\nDistribusi {name}")
    print(y.value_counts())
    print(y.value_counts(normalize=True))

check_distribution("TRAIN", y_train)
check_distribution("VAL", y_val)
check_distribution("TEST", y_test)


Distribusi TRAIN
Class
0    199020
1       344
Name: count, dtype: int64
Class
0    0.998275
1    0.001725
Name: proportion, dtype: float64

Distribusi VAL
Class
0    42647
1       74
Name: count, dtype: int64
Class
0    0.998268
1    0.001732
Name: proportion, dtype: float64

Distribusi TEST
Class
0    42648
1       74
Name: count, dtype: int64
Class
0    0.998268
1    0.001732
Name: proportion, dtype: float64


In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols)
    ]
)

# FIT hanya TRAIN
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

print("Train shape:", X_train_prep.shape)
print("Val shape:", X_val_prep.shape)
print("Test shape:", X_test_prep.shape)

Train shape: (199364, 30)
Val shape: (42721, 30)
Test shape: (42722, 30)
